In [2]:
import math
import numpy as np
import random

# --- 1. Problem Setup (Depot + 8 Customers) ---
# Coordinates: index 0 is the Depot, indices 1..8 are Customers
COORDS = np.array([
    [20, 20],  # 0: Depot
    [10, 10],  # 1: Customer 1
    [10, 30],  # 2: Customer 2
    [30, 30],  # 3: Customer 3
    [30, 10],  # 4: Customer 4
    [15, 15],  # 5: Customer 5
    [25, 25],  # 6: Customer 6
    [10, 20],  # 7: Customer 7
    [15, 40]   # 8: Customer 8
])

# Demand at each node (Depot demand = 0)
DEMANDS = [0, 15, 20, 25, 10, 15, 20, 10, 15]
VEHICLE_CAPACITY = 45

# Precompute Euclidean Distance Matrix
NUM_NODES = len(COORDS)
DIST_MATRIX = np.zeros((NUM_NODES, NUM_NODES))
for i in range(NUM_NODES):
    for j in range(NUM_NODES):
        DIST_MATRIX[i][j] = np.linalg.norm(COORDS[i] - COORDS[j])


# --- 2. VRP Helper Functions ---
def decode_routes(permutation):
    """Splits a customer permutation into feasible vehicle routes based on capacity."""
    routes = []
    current_route = [0]
    current_load = 0

    for customer in permutation:
        demand = DEMANDS[customer]
        if current_load + demand <= VEHICLE_CAPACITY:
            current_route.append(customer)
            current_load += demand
        else:
            current_route.append(0)  # Return to depot
            routes.append(current_route)
            current_route = [0, customer]
            current_load = demand

    current_route.append(0)  # Final return to depot
    routes.append(current_route)
    return routes


def calculate_fitness(permutation):
    """Calculates total distance traveled across all vehicle routes."""
    routes = decode_routes(permutation)
    total_dist = 0.0
    for route in routes:
        for i in range(len(route) - 1):
            total_dist += DIST_MATRIX[route[i]][route[i + 1]]
    return total_dist


# --- 3. Discrete L	vy Flight Operator ---
def levy_flight_step(beta=1.5):
    """Generates a L	vy step size using Mantegna's algorithm."""
    sigma_u = (
        math.gamma(1 + beta) * math.sin(math.pi * beta / 2) /
        (math.gamma((1 + beta) / 2) * beta * (2 ** ((beta - 1) / 2)))
    ) ** (1 / beta)
    u = np.random.normal(0, sigma_u)
    v = np.random.normal(0, 1)
    step = u / (abs(v) ** (1 / beta))
    return abs(step)


def discrete_levy_mutation(permutation):
    """Applies small swaps for small L	vy steps and large 2-opt/shuffles for big jumps."""
    new_perm = permutation.copy()
    step = levy_flight_step()
    n = len(new_perm)

    if step < 1.0:
        # Small local step: Swap two adjacent/random customers
        i, j = random.sample(range(n), 2)
        new_perm[i], new_perm[j] = new_perm[j], new_perm[i]
    elif step < 2.5:
        # Medium step: 2-opt segment reversal
        i, j = sorted(random.sample(range(n), 2))
        new_perm[i:j+1] = reversed(new_perm[i:j+1])
    else:
        # Large L	vy jump: Double-bridge / multi-point shuffle
        num_swaps = min(n, int(step))
        for _ in range(num_swaps):
            i, j = random.sample(range(n), 2)
            new_perm[i], new_perm[j] = new_perm[j], new_perm[i]

    return new_perm


# --- 4. Cuckoo Search Algorithm for VRP ---
def cuckoo_search_vrp(n_nests=20, pa=0.25, max_iter=300):
    customers = list(range(1, NUM_NODES))

    # Initialize n host nests with random customer permutations
    nests = [random.sample(customers, len(customers)) for _ in range(n_nests)]
    fitness = [calculate_fitness(nest) for nest in nests]

    # Track best nest
    best_idx = int(np.argmin(fitness))
    best_nest = nests[best_idx].copy()
    best_fitness = fitness[best_idx]

    for t in range(max_iter):
        # Generate a new solution via L	vy flight from a random nest
        i = random.randint(0, n_nests - 1)
        new_nest = discrete_levy_mutation(nests[i])
        new_fitness = calculate_fitness(new_nest)

        # Choose a random host nest j to compare
        j = random.randint(0, n_nests - 1)
        if new_fitness < fitness[j]:
            nests[j] = new_nest
            fitness[j] = new_fitness

        # Abandon a fraction (Pa) of the worst nests and build new ones
        num_abandon = max(1, int(pa * n_nests))
        sorted_indices = np.argsort(fitness)  # Ascending order (lowest distance first)

        for idx in sorted_indices[-num_abandon:]:
            nests[idx] = random.sample(customers, len(customers))
            fitness[idx] = calculate_fitness(nests[idx])

        # Update global best solution
        current_best_idx = int(np.argmin(fitness))
        if fitness[current_best_idx] < best_fitness:
            best_fitness = fitness[current_best_idx]
            best_nest = nests[current_best_idx].copy()

    return decode_routes(best_nest), best_fitness


# --- 5. Run the Solver ---
if __name__ == "__main__":
    random.seed(42)
    np.random.seed(42)

    optimal_routes, min_distance = cuckoo_search_vrp(n_nests=25, pa=0.25, max_iter=500)

    print(f"Minimum Total Distance: {min_distance:.2f}")
    for v_idx, route in enumerate(optimal_routes, 1):
        route_demand = sum(DEMANDS[node] for node in route)
        print(f"Vehicle {v_idx}: {route} | Load: {route_demand}/{VEHICLE_CAPACITY}")

Minimum Total Distance: 128.36
Vehicle 1: [0, 4, 1, 5, 0] | Load: 40/45
Vehicle 2: [0, 3, 6, 0] | Load: 45/45
Vehicle 3: [0, 8, 2, 7, 0] | Load: 45/45
